# DL Experiments: BERT4Rec small

Ноутбук:
- строит sequence dataset из `data/datamart/agg`;
- обучает BERT4Rec в fast/proper режиме;
- генерирует candidates `dl_bert4rec_small` для `ranker_train`, `val`, `test`;
- делает safe merge с текущим non-DL pool `merged`;
- считает validation metrics, unique positives и quota ablation.

In [1]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import shutil
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm, trange

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    import hnswlib
    HAS_HNSWLIB = True
except Exception:
    HAS_HNSWLIB = False

try:
    import faiss
    HAS_FAISS = True
except Exception:
    HAS_FAISS = False

In [2]:
# -----------------------------
# Config
# -----------------------------

def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").exists():
            return p
    return start.resolve()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = DATA_DIR / "datamart"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
PROCESSED_DIR = DATA_DIR / "processed" / "dl_candidate_generators_bert4rec_small"
MODELS_DIR = PROJECT_ROOT / "models" / "dl_candidate_generators_bert4rec_small"


@dataclass(frozen=True)
class BERT4RecPipelineConfig:
    # Same time split as candidate/reranker notebooks.
    ranker_train_context_start: date = date(2025, 4, 18)
    ranker_train_context_end: date = date(2025, 6, 3)
    ranker_train_target_start: date = date(2025, 6, 4)
    ranker_train_target_end: date = date(2025, 6, 17)

    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    random_state: int = 42
    n_shards: int = 128
    metric_ks: tuple[int, ...] = (20, 50, 100, 200, 300, 700, 900, 1000)

    # Source/output names.
    source_name: str = "dl_bert4rec_small"
    base_candidate_name: str = "merged"
    output_merged_name: str = "merged_with_bert4rec_small"

    # Main run controls.
    force_retrain_bert4rec: bool = False
    train_bert4rec_if_missing: bool = True
    rebuild_sequences: bool = False
    rebuild_model: bool = False
    rebuild_ann_index: bool = False
    rebuild_sources: bool = False
    rebuild_merged: bool = True
    evaluate_val: bool = True

    # One checkpoint is trained on validation context and reused for all inference splits.
    # This mirrors the final DSSM setup and avoids training three separate Transformer models.
    checkpoint_split: str = "val"
    splits_to_build_sources: tuple[str, ...] = ("ranker_train", "val", "test")
    splits_to_merge: tuple[str, ...] = ("ranker_train", "val", "test")

    # Vocab / sequences.
    max_item_vocab: int = 1_200_000
    min_item_weight: float = 1.0
    sequence_max_len: int = 64
    min_sequence_len: int = 2
    max_train_examples: int | None = 5_000_000
    max_targets_per_user: int = 12
    # BERT4Rec masked-item pretraining. One masked target is sampled per training example.
    mask_token_strategy: str = "single_position"
    mask_random_item_prob: float = 0.10
    mask_keep_original_prob: float = 0.10
    train_valid_fraction: float = 0.01
    max_valid_examples: int = 50_000

    # Interaction weights.
    weak_max_cap: float = 20.0
    time_decay_enabled: bool = True
    time_decay_tau_days: float = 30.0
    time_decay_min_factor: float = 0.10
    event_weight_page_view: float = 1.0
    event_weight_view_description: float = 2.0
    event_weight_review_view: float = 3.0
    event_weight_favorite: float = 20.0
    event_weight_to_cart: float = 100.0
    order_weight: float = 100.0

    # BERT4Rec architecture.
    embedding_dim: int = 192
    num_heads: int = 4
    num_layers: int = 2
    dropout: float = 0.15
    use_catalog_embeddings: bool = True
    catalog_dim: int = 64
    cat_l0_dim: int = 32
    use_event_type_embeddings: bool = True
    event_type_dim: int = 32
    num_event_types: int = 6  # 0=padding, 1=view, 2=description/review, 3=favorite, 4=cart, 5=order
    use_recency_embeddings: bool = True
    recency_dim: int = 32
    recency_bucket_edges_days: tuple[int, ...] = (0, 1, 2, 3, 7, 14, 30, 60, 120, 365)
    temperature_init: float = 0.07

    # Training.
    pretrain_epochs: int = 2
    finetune_on_orders: bool = True
    finetune_epochs: int = 2
    batch_size: int = 2048
    finetune_batch_size: int = 2048
    lr: float = 5e-4
    finetune_lr: float = 1.0e-4
    weight_decay: float = 1e-5
    grad_clip_norm: float = 5.0
    scheduler_pct_start: float = 0.05
    use_amp: bool = True
    allow_tf32: bool = True
    num_workers: int = 0
    pin_memory: bool = True
    log_every_steps: int = 50
    eval_every_epoch: bool = True

    # Loss details.
    multipositive_same_user: bool = True
    popularity_correction_beta: float = 0.05
    logq_smoothing: float = 1.0

    # Retrieval calibration. Applied after ANN oversampling and before taking top candidates.
    # This is intentionally light: BERT4Rec stays a sequential retriever, popularity only breaks weak ties.
    bert4rec_popularity_prior_weight: float = 0.03
    bert4rec_user_cat_l0_prior_weight: float = 0.06
    bert4rec_user_catalog_prior_weight: float = 0.03
    bert4rec_raw_score_weight: float = 1.0

    # ANN retrieval.
    ann_backend: str = "auto"  # auto | hnswlib | faiss | torch | numpy
    ann_oversample: int = 1200
    candidates_per_user: int = 300
    filter_seen_context_items: bool = False
    retrieval_user_batch_size: int = 2048
    retrieval_flush_rows: int = 5_000_000

    hnsw_m: int = 24
    hnsw_ef_construction: int = 120
    hnsw_ef_search: int = 300
    hnsw_add_batch_size: int = 100_000
    hnsw_num_threads: int = max(1, min(8, os.cpu_count() or 1))

    faiss_index_type: str = "flat"  # flat | ivf_flat
    faiss_use_gpu: bool = False
    faiss_nlist: int = 4096
    faiss_nprobe: int = 32
    faiss_train_sample_size: int = 1_000_000

    # Safe merge.
    base_keep_per_user: int = 700
    bert4rec_keep_per_user: int = 300
    merged_total_keep_per_user: int = 1000

CFG = BERT4RecPipelineConfig()

# Fast/proper experiment defaults are set directly in the config above.
# If this source was partially built and you want a clean run, set:
# CFG = replace(CFG, force_retrain_bert4rec=True, rebuild_model=True, rebuild_sources=True, rebuild_ann_index=True, rebuild_merged=True)

# For intentional full retrain:
# CFG = replace(CFG, force_retrain_bert4rec=True, rebuild_model=True, rebuild_sources=True, rebuild_ann_index=True)

In [3]:
# -----------------------------
# I/O helpers
# -----------------------------

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return lf.collect_schema().names()
    except Exception:
        return lf.columns


def iter_parquet_files(path: Path):
    if path.is_file():
        yield path
    elif path.exists():
        yield from sorted(path.rglob("*.parquet"))


def scan_parquet(path: Path, **kwargs) -> pl.LazyFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pl.scan_parquet(str(path), **kwargs)


def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def clean_output_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def candidate_dir_looks_complete(path: Path) -> bool:
    # Some DL sources may create many files per shard; a _meta.json + parquet is enough for reuse.
    return path.exists() and (path / "_meta.json").exists() and any(path.rglob("*.parquet"))


def shard_dir(path: Path, shard_id: int) -> Path:
    return path / f"user_shard={shard_id}"


def scan_candidate_shard_or_none(path: Path, shard_id: int) -> pl.LazyFrame | None:
    sdir = shard_dir(path, shard_id)
    if not sdir.exists():
        return None
    files = list(sdir.rglob("*.parquet"))
    if not files:
        return None
    return pl.scan_parquet(str(sdir / "*.parquet"))


def events_path(level: str) -> Path:
    return DATAMART_DIR / "agg" / "events" / level


def orders_path(level: str) -> Path:
    return DATAMART_DIR / "agg" / "orders" / level


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(events_path(level), hive_partitioning=True)
    if "event_date" not in lazy_columns(lf):
        raise ValueError(f"{events_path(level)} must contain event_date")
    return lf.filter((pl.col("event_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("event_date").cast(pl.Date) <= pl.lit(end)))


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(orders_path(level), hive_partitioning=True)
    if "created_date" not in lazy_columns(lf):
        raise ValueError(f"{orders_path(level)} must contain created_date")
    return lf.filter((pl.col("created_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("created_date").cast(pl.Date) <= pl.lit(end)))


def ensure_numeric_cols(lf: pl.LazyFrame, cols: list[str]) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    exprs = []
    for c in cols:
        if c in existing:
            exprs.append(pl.col(c).fill_null(0).cast(pl.Float32).alias(c))
        else:
            exprs.append(pl.lit(0.0, dtype=pl.Float32).alias(c))
    return lf.with_columns(exprs)


def split_spec(split_name: str) -> tuple[date, date, date, date]:
    if split_name == "ranker_train":
        return CFG.ranker_train_context_start, CFG.ranker_train_context_end, CFG.ranker_train_target_start, CFG.ranker_train_target_end
    if split_name == "val":
        return CFG.val_context_start, CFG.val_context_end, CFG.val_target_start, CFG.val_target_end
    if split_name == "test":
        return CFG.full_context_start, CFG.full_context_end, CFG.test_target_start, CFG.test_target_end
    raise ValueError(f"Unknown split_name={split_name!r}")


def filter_lf_by_users(lf: pl.LazyFrame, user_ids: np.ndarray | None) -> pl.LazyFrame:
    if user_ids is None:
        return lf
    if len(user_ids) == 0:
        return lf.filter(pl.lit(False))
    users_lf = pl.DataFrame({"user_id": np.asarray(user_ids, dtype=np.int64)}).lazy()
    return lf.join(users_lf, on="user_id", how="inner")


def weight_values() -> dict[str, float]:
    return {
        "page_view": CFG.event_weight_page_view,
        "view_description": CFG.event_weight_view_description,
        "review_view": CFG.event_weight_review_view,
        "favorite": CFG.event_weight_favorite,
        "to_cart": CFG.event_weight_to_cart,
        "order": CFG.order_weight,
    }


def time_decay_expr(date_col: str, context_end: date) -> pl.Expr:
    if not CFG.time_decay_enabled:
        return pl.lit(1.0, dtype=pl.Float32)
    days = (pl.lit(context_end).cast(pl.Date) - pl.col(date_col).cast(pl.Date)).dt.total_days().cast(pl.Float32)
    return pl.max_horizontal(
        pl.lit(float(CFG.time_decay_min_factor), dtype=pl.Float32),
        (-days / float(CFG.time_decay_tau_days)).exp().cast(pl.Float32),
    )


def recency_bucket_np(dates, context_end: date) -> np.ndarray:
    """1-based recency buckets for sequence tokens; 0 is reserved for padding."""
    dt = pd.to_datetime(dates)
    days = (pd.Timestamp(context_end) - dt).dt.days.to_numpy(dtype=np.int32)
    edges = np.asarray(CFG.recency_bucket_edges_days, dtype=np.int32)
    # np.digitize returns 0..len(edges); add 1 to keep 0 for padding.
    return (np.digitize(days, edges, right=True) + 1).astype(np.int16)


ensure_dir(REPORTS_DIR)
ensure_dir(PROCESSED_DIR)
ensure_dir(MODELS_DIR)

seed_everything(CFG.random_state)

if CFG.allow_tf32 and torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("HAS_HNSWLIB:", HAS_HNSWLIB, "HAS_FAISS:", HAS_FAISS)
print("source:", CFG.source_name, "output:", CFG.output_merged_name)

PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
DEVICE: cuda
GPU: NVIDIA GeForce RTX 3090
HAS_HNSWLIB: True HAS_FAISS: True
source: dl_bert4rec_small output: merged_with_bert4rec_small


In [4]:
# -----------------------------
# Sequence construction
# -----------------------------

def build_interactions_lf(start: date, end: date, base_users: np.ndarray | None = None) -> pl.LazyFrame:
    """Weighted user-item interactions inside context window.

    v3 keeps a dominant event_type_id per (user,item,date), so BERT4Rec can distinguish:
    1=view, 2=description/review, 3=favorite, 4=cart, 5=order.
    """
    w = weight_values()

    event_cols = ["cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
    e = ensure_numeric_cols(read_events_lf("user_item", start, end), event_cols)
    e = filter_lf_by_users(e, base_users)

    event_weight = (
        pl.col("cnt_page_view").clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["page_view"]
        + pl.col("cnt_view_description").clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["view_description"]
        + pl.col("cnt_review_view").clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["review_view"]
        + (pl.col("cnt_favorite").fill_null(0) > 0).cast(pl.Float32) * w["favorite"]
        + (pl.col("cnt_to_cart").fill_null(0) > 0).cast(pl.Float32) * w["to_cart"]
    ) * time_decay_expr("event_date", end)

    event_type_id = (
        pl.when(pl.col("cnt_to_cart").fill_null(0) > 0).then(pl.lit(4))
        .when(pl.col("cnt_favorite").fill_null(0) > 0).then(pl.lit(3))
        .when((pl.col("cnt_review_view").fill_null(0) > 0) | (pl.col("cnt_view_description").fill_null(0) > 0)).then(pl.lit(2))
        .otherwise(pl.lit(1))
        .cast(pl.Int16)
    )

    e = (
        e.with_columns(
            event_weight.alias("weight"),
            pl.col("event_date").cast(pl.Date).alias("dt"),
            event_type_id.alias("event_type_id"),
        )
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32), "event_type_id")
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    o = ensure_numeric_cols(read_orders_lf("user_item", start, end), ["cnt_orders"])
    o = filter_lf_by_users(o, base_users)
    order_weight = (
        pl.col("cnt_orders").fill_null(0).cast(pl.Float32) * w["order"] * time_decay_expr("created_date", end)
    ).cast(pl.Float32)
    o = (
        o.with_columns(
            order_weight.alias("weight"),
            pl.col("created_date").cast(pl.Date).alias("dt"),
            pl.lit(5, dtype=pl.Int16).alias("event_type_id"),
        )
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32), "event_type_id")
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    return (
        pl.concat([e, o], how="diagonal_relaxed")
        .group_by(["user_id", "item_id", "dt"])
        .agg(
            pl.sum("weight").cast(pl.Float32).alias("weight"),
            pl.max("event_type_id").cast(pl.Int16).alias("event_type_id"),
        )
        .filter(pl.col("weight") > 0)
    )

def build_item_vocab(interactions: pl.DataFrame, max_items: int) -> tuple[dict[int, int], np.ndarray, np.ndarray]:
    pop = (
        interactions.group_by("item_id")
        .agg(pl.sum("weight").alias("item_weight"))
        .sort(["item_weight", "item_id"], descending=[True, False])
        .head(max_items)
    )
    item_ids = pop["item_id"].cast(pl.Int64).to_numpy()
    weights = pop["item_weight"].cast(pl.Float32).to_numpy()
    item_id_by_idx = np.concatenate([np.array([0], dtype=np.int64), item_ids.astype(np.int64)])
    item_weight_by_idx = np.concatenate([np.array([0.0], dtype=np.float32), weights.astype(np.float32)])
    item2idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
    return item2idx, item_id_by_idx, item_weight_by_idx


def interactions_to_sequences(
    interactions: pl.DataFrame,
    item2idx: dict[int, int],
    context_end: date,
) -> tuple[np.ndarray, list[np.ndarray], list[np.ndarray], list[np.ndarray], dict[int, np.ndarray], dict[int, np.ndarray], dict[int, np.ndarray]]:
    # Pandas groupby is acceptable here because this stage is cached.
    item_keys = list(item2idx.keys())
    df = (
        interactions
        .filter(pl.col("item_id").is_in(item_keys))
        .sort(["user_id", "dt", "weight", "item_id"], descending=[False, False, True, False])
        .select("user_id", "item_id", "dt", "weight", "event_type_id")
        .to_pandas()
    )
    df["item_idx"] = df["item_id"].map(item2idx).astype(np.int32)
    df["event_type_id"] = df["event_type_id"].fillna(1).astype(np.int16).clip(1, CFG.num_event_types - 1)
    df["recency_bucket"] = recency_bucket_np(df["dt"], context_end)

    user_ids: list[int] = []
    seqs: list[np.ndarray] = []
    event_seqs: list[np.ndarray] = []
    recency_seqs: list[np.ndarray] = []
    user_seq: dict[int, np.ndarray] = {}
    user_event_seq: dict[int, np.ndarray] = {}
    user_recency_seq: dict[int, np.ndarray] = {}

    for user_id, g in tqdm(df.groupby("user_id", sort=False), desc="Build user sequences"):
        seq = g["item_idx"].to_numpy(dtype=np.int32)
        if len(seq) < CFG.min_sequence_len:
            continue
        ev = g["event_type_id"].to_numpy(dtype=np.int16)
        rb = g["recency_bucket"].to_numpy(dtype=np.int16)
        user_id = int(user_id)
        user_ids.append(user_id)
        seqs.append(seq)
        event_seqs.append(ev)
        recency_seqs.append(rb)
        user_seq[user_id] = seq
        user_event_seq[user_id] = ev
        user_recency_seq[user_id] = rb

    del df
    gc.collect()
    return (
        np.asarray(user_ids, dtype=np.int64),
        seqs,
        event_seqs,
        recency_seqs,
        user_seq,
        user_event_seq,
        user_recency_seq,
    )

def sequence_cache_dir(split_name: str) -> Path:
    return PROCESSED_DIR / "sequences" / split_name


def load_or_build_training_sequences(split_name: str):
    start, end, _, _ = split_spec(split_name)
    cache = sequence_cache_dir(split_name)
    ensure_dir(cache)

    item_map_path = cache / "item_id_by_idx.npy"
    item_weight_path = cache / "item_weight_by_idx.npy"
    user_ids_path = cache / "user_ids.npy"
    seqs_path = cache / "seqs.npy"
    events_path = cache / "event_types.npy"
    recency_path = cache / "recency_buckets.npy"
    meta_path = cache / "meta.json"

    if (
        item_map_path.exists()
        and item_weight_path.exists()
        and user_ids_path.exists()
        and seqs_path.exists()
        and events_path.exists()
        and recency_path.exists()
        and meta_path.exists()
        and not CFG.rebuild_sequences
    ):
        item_id_by_idx = np.load(item_map_path)
        item_weight_by_idx = np.load(item_weight_path)
        seq_user_ids = np.load(user_ids_path).astype(np.int64)
        seqs = [np.asarray(x, dtype=np.int32) for x in np.load(seqs_path, allow_pickle=True).tolist()]
        event_seqs = [np.asarray(x, dtype=np.int16) for x in np.load(events_path, allow_pickle=True).tolist()]
        recency_seqs = [np.asarray(x, dtype=np.int16) for x in np.load(recency_path, allow_pickle=True).tolist()]
        item2idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
        user_seq = {int(u): seqs[i] for i, u in enumerate(seq_user_ids)}
        user_event_seq = {int(u): event_seqs[i] for i, u in enumerate(seq_user_ids)}
        user_recency_seq = {int(u): recency_seqs[i] for i, u in enumerate(seq_user_ids)}
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        print(f"{split_name}: training sequences loaded -> {rel(cache)}, users={len(seq_user_ids):,}, items={len(item_id_by_idx)-1:,}")
        return item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, event_seqs, recency_seqs, user_seq, user_event_seq, user_recency_seq, meta

    print(f"{split_name}: building context interactions {start}..{end}")
    interactions = collect_streaming(build_interactions_lf(start, end))
    print(f"{split_name}: interactions rows={interactions.height:,}")

    item2idx, item_id_by_idx, item_weight_by_idx = build_item_vocab(interactions, CFG.max_item_vocab)
    seq_user_ids, seqs, event_seqs, recency_seqs, user_seq, user_event_seq, user_recency_seq = interactions_to_sequences(interactions, item2idx, end)

    np.save(item_map_path, item_id_by_idx)
    np.save(item_weight_path, item_weight_by_idx)
    np.save(user_ids_path, seq_user_ids)
    np.save(seqs_path, np.asarray(seqs, dtype=object), allow_pickle=True)
    np.save(events_path, np.asarray(event_seqs, dtype=object), allow_pickle=True)
    np.save(recency_path, np.asarray(recency_seqs, dtype=object), allow_pickle=True)
    meta = {
        "split": split_name,
        "context_start": str(start),
        "context_end": str(end),
        "users": int(len(seq_user_ids)),
        "items": int(len(item_id_by_idx) - 1),
        "sequence_has_event_type": True,
        "sequence_has_recency_bucket": True,
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(CFG),
    }
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    del interactions
    gc.collect()
    print(f"{split_name}: training sequences saved -> {rel(cache)}")
    return item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, event_seqs, recency_seqs, user_seq, user_event_seq, user_recency_seq, meta

def resolve_base_candidate_dir(split: str) -> Path:
    path = CANDIDATES_DIR / CFG.base_candidate_name / split
    if not path.exists() or not any(path.rglob("*.parquet")):
        raise FileNotFoundError(f"Base candidate dir not found: {path}")
    return path


def read_base_users(split: str) -> np.ndarray:
    cache = PROCESSED_DIR / "base_users"
    ensure_dir(cache)
    out = cache / f"base_users_{split}.npy"
    if out.exists() and not CFG.rebuild_sequences:
        arr = np.load(out).astype(np.int64)
        print(f"{split}: base users loaded -> {rel(out)}, users={len(arr):,}")
        return arr

    base_dir = resolve_base_candidate_dir(split)
    users = []
    for shard_id in trange(CFG.n_shards, desc=f"Read base users {split}"):
        lf = scan_candidate_shard_or_none(base_dir, shard_id)
        if lf is None:
            continue
        part = collect_streaming(lf.select(pl.col("user_id").cast(pl.Int64).unique()))
        users.append(part["user_id"].to_numpy())
    if not users:
        raise RuntimeError(f"No base users read for split={split}")
    arr = np.unique(np.concatenate(users).astype(np.int64))
    np.save(out, arr)
    print(f"{split}: base users saved -> {rel(out)}, users={len(arr):,}")
    return arr


def build_sequences_with_checkpoint_vocab(
    split_name: str,
    item2idx: dict[int, int],
) -> tuple[np.ndarray, dict[int, np.ndarray], dict[int, np.ndarray], dict[int, np.ndarray]]:
    cache = PROCESSED_DIR / "inference_sequences" / split_name
    ensure_dir(cache)
    user_ids_path = cache / "user_ids.npy"
    seqs_path = cache / "seqs.npy"
    events_path = cache / "event_types.npy"
    recency_path = cache / "recency_buckets.npy"
    meta_path = cache / "meta.json"

    if user_ids_path.exists() and seqs_path.exists() and events_path.exists() and recency_path.exists() and meta_path.exists() and not CFG.rebuild_sequences:
        user_ids = np.load(user_ids_path).astype(np.int64)
        seqs = [np.asarray(x, dtype=np.int32) for x in np.load(seqs_path, allow_pickle=True).tolist()]
        event_seqs = [np.asarray(x, dtype=np.int16) for x in np.load(events_path, allow_pickle=True).tolist()]
        recency_seqs = [np.asarray(x, dtype=np.int16) for x in np.load(recency_path, allow_pickle=True).tolist()]
        user_seq = {int(u): seqs[i] for i, u in enumerate(user_ids)}
        user_event_seq = {int(u): event_seqs[i] for i, u in enumerate(user_ids)}
        user_recency_seq = {int(u): recency_seqs[i] for i, u in enumerate(user_ids)}
        print(f"{split_name}: inference sequences loaded -> {rel(cache)}, users={len(user_ids):,}")
        return user_ids, user_seq, user_event_seq, user_recency_seq

    base_users = read_base_users(split_name)
    start, end, _, _ = split_spec(split_name)
    item_map = pl.DataFrame({
        "item_id": np.fromiter(item2idx.keys(), dtype=np.int64),
        "item_idx": np.fromiter(item2idx.values(), dtype=np.int32),
    }).lazy()

    print(f"{split_name}: build inference sequences with checkpoint vocab, context={start}..{end}, base_users={len(base_users):,}")
    lf = (
        build_interactions_lf(start, end, base_users=base_users)
        .join(item_map, on="item_id", how="inner")
        .select("user_id", "item_idx", "dt", "weight", "event_type_id")
        .sort(["user_id", "dt", "weight", "item_idx"], descending=[False, False, True, False])
    )
    df = collect_streaming(lf).to_pandas()
    if df.empty:
        raise RuntimeError(f"No sequences built for split={split_name}")
    df["recency_bucket"] = recency_bucket_np(df["dt"], end)

    user_ids: list[int] = []
    seqs: list[np.ndarray] = []
    event_seqs: list[np.ndarray] = []
    recency_seqs: list[np.ndarray] = []
    user_seq: dict[int, np.ndarray] = {}
    user_event_seq: dict[int, np.ndarray] = {}
    user_recency_seq: dict[int, np.ndarray] = {}

    for user_id, g in tqdm(df.groupby("user_id", sort=False), desc=f"Build inference user sequences {split_name}"):
        seq = g["item_idx"].to_numpy(dtype=np.int32)[-CFG.sequence_max_len:]
        ev = g["event_type_id"].fillna(1).astype(np.int16).to_numpy()[-CFG.sequence_max_len:]
        rb = g["recency_bucket"].astype(np.int16).to_numpy()[-CFG.sequence_max_len:]
        if len(seq) == 0:
            continue
        user_id = int(user_id)
        user_ids.append(user_id)
        seqs.append(seq)
        event_seqs.append(ev)
        recency_seqs.append(rb)
        user_seq[user_id] = seq
        user_event_seq[user_id] = ev
        user_recency_seq[user_id] = rb

    user_ids_arr = np.asarray(user_ids, dtype=np.int64)
    np.save(user_ids_path, user_ids_arr)
    np.save(seqs_path, np.asarray(seqs, dtype=object), allow_pickle=True)
    np.save(events_path, np.asarray(event_seqs, dtype=object), allow_pickle=True)
    np.save(recency_path, np.asarray(recency_seqs, dtype=object), allow_pickle=True)
    meta_path.write_text(json.dumps({
        "split": split_name,
        "context_start": str(start),
        "context_end": str(end),
        "base_users": int(len(base_users)),
        "users_with_checkpoint_vocab_history": int(len(user_ids_arr)),
        "sequence_max_len": int(CFG.sequence_max_len),
        "sequence_has_event_type": True,
        "sequence_has_recency_bucket": True,
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{split_name}: inference sequences saved -> {rel(cache)}, users={len(user_ids_arr):,}")
    return user_ids_arr, user_seq, user_event_seq, user_recency_seq

In [5]:
# -----------------------------
# Item side category arrays
# -----------------------------

def raw_items_data_path() -> Path | None:
    candidates = [
        DATA_DIR / "raw" / "ml_ozon_recsys_train_final_apparel_items_data",
        DATA_DIR / "raw" / "items",
        DATA_DIR / "raw" / "items.parquet",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def load_item_catalog_mapping() -> pd.DataFrame:
    cache = PROCESSED_DIR / "item_catalog_mapping.parquet"
    if cache.exists():
        return pd.read_parquet(cache)

    raw_path = raw_items_data_path()
    if raw_path is None:
        warnings.warn("Raw items table not found. BERT4Rec will run without category side features.")
        return pd.DataFrame(columns=["item_id", "catalogid"])

    parts = []
    for f in tqdm(list(iter_parquet_files(raw_path)), desc="Read item catalog mapping"):
        try:
            cols = pl.scan_parquet(str(f)).collect_schema().names()
            has_item = "item_id" in cols
            has_catalog = "catalogid" in cols
            if not has_item or not has_catalog:
                continue
            df = pl.read_parquet(f, columns=["item_id", "catalogid"])
            parts.append(df)
        except Exception:
            continue
    if not parts:
        return pd.DataFrame(columns=["item_id", "catalogid"])
    out = pl.concat(parts, how="diagonal_relaxed").select(
        pl.col("item_id").cast(pl.Int64),
        pl.col("catalogid").cast(pl.Int64),
    ).unique().to_pandas()
    ensure_dir(cache.parent)
    out.to_parquet(cache, index=False)
    return out


def build_item_side_arrays(item_id_by_idx: np.ndarray, out_dir: Path, split_name: str) -> dict:
    ensure_dir(out_dir)
    side_path = out_dir / f"item_side_{split_name}.npz"
    meta_path = out_dir / f"item_side_{split_name}.json"

    if side_path.exists() and meta_path.exists() and not CFG.rebuild_sequences:
        data = np.load(side_path)
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        return {
            "catalog_idx_by_item_idx": data["catalog_idx_by_item_idx"].astype(np.int64),
            "cat_l0_idx_by_item_idx": data["cat_l0_idx_by_item_idx"].astype(np.int64),
            "num_catalogs": int(meta["num_catalogs"]),
            "num_cat_l0": int(meta["num_cat_l0"]),
        }

    mapping = load_item_catalog_mapping()
    item_to_pos = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}

    catalog_raw = np.zeros(len(item_id_by_idx), dtype=np.int64)
    if not mapping.empty:
        mapping = mapping.dropna()
        for item_id, catalogid in zip(mapping["item_id"].astype(np.int64), mapping["catalogid"].astype(np.int64)):
            pos = item_to_pos.get(int(item_id))
            if pos is not None:
                catalog_raw[pos] = int(catalogid)

    # Coarse category derived from catalog id. It is stable enough for a side signal.
    cat_l0_raw = np.where(catalog_raw > 0, catalog_raw // 1000, 0).astype(np.int64)

    unique_catalogs = sorted(int(x) for x in np.unique(catalog_raw) if int(x) > 0)
    unique_l0 = sorted(int(x) for x in np.unique(cat_l0_raw) if int(x) > 0)
    catalog2idx = {c: i + 1 for i, c in enumerate(unique_catalogs)}
    l02idx = {c: i + 1 for i, c in enumerate(unique_l0)}

    catalog_idx = np.zeros_like(catalog_raw, dtype=np.int64)
    cat_l0_idx = np.zeros_like(cat_l0_raw, dtype=np.int64)
    for i in range(1, len(item_id_by_idx)):
        c = int(catalog_raw[i])
        l0 = int(cat_l0_raw[i])
        if c > 0:
            catalog_idx[i] = catalog2idx.get(c, 0)
        if l0 > 0:
            cat_l0_idx[i] = l02idx.get(l0, 0)

    np.savez_compressed(side_path, catalog_idx_by_item_idx=catalog_idx, cat_l0_idx_by_item_idx=cat_l0_idx)
    meta = {"num_catalogs": len(unique_catalogs) + 1, "num_cat_l0": len(unique_l0) + 1}
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{split_name}: item side arrays saved -> {rel(side_path)}")
    return {
        "catalog_idx_by_item_idx": catalog_idx,
        "cat_l0_idx_by_item_idx": cat_l0_idx,
        "num_catalogs": len(unique_catalogs) + 1,
        "num_cat_l0": len(unique_l0) + 1,
    }

In [6]:
# -----------------------------
# Datasets
# -----------------------------

def make_next_item_examples(seqs: list[np.ndarray], max_examples: int | None, max_targets_per_user: int, seed: int) -> np.ndarray:
    """Build (sequence_index, target_position) examples.

    For BERT4Rec pretraining the target position is masked inside the observed context.
    Unlike SASRec, the model may use both left and right context around the masked item.
    """
    examples = []
    for uidx, seq in enumerate(seqs):
        n = len(seq)
        if n < CFG.min_sequence_len:
            continue
        positions = np.arange(1, n, dtype=np.int32)
        if len(positions) > max_targets_per_user:
            positions = positions[-max_targets_per_user:]
        for pos in positions:
            examples.append((uidx, int(pos)))
    if not examples:
        raise RuntimeError("No masked-item training examples built")
    arr = np.asarray(examples, dtype=np.int32)
    rng = np.random.default_rng(seed)
    if max_examples is not None and len(arr) > max_examples:
        keep = rng.choice(len(arr), size=max_examples, replace=False)
        arr = arr[keep]
    rng.shuffle(arr)
    return arr


def split_train_valid_examples(examples: np.ndarray, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    idx = np.arange(len(examples))
    rng.shuffle(idx)
    n_valid = min(CFG.max_valid_examples, max(1, int(len(examples) * CFG.train_valid_fraction)))
    valid_idx = idx[:n_valid]
    train_idx = idx[n_valid:]
    return examples[train_idx], examples[valid_idx]


def _pad_right_aligned(values: np.ndarray, max_len: int, dtype=np.int64) -> np.ndarray:
    out = np.zeros(max_len, dtype=dtype)
    if values is not None and len(values) > 0:
        vals = np.asarray(values[-max_len:], dtype=dtype)
        out[-len(vals):] = vals
    return out


def _bert_window_around_position(seq: np.ndarray, pos: int, max_len: int) -> tuple[int, int]:
    """Return [start, end) window of length <= max_len containing pos."""
    n = len(seq)
    if n <= max_len:
        return 0, n
    left_budget = max(1, int(max_len * 0.75))
    start = max(0, pos - left_budget)
    end = start + max_len
    if end > n:
        end = n
        start = max(0, end - max_len)
    if not (start <= pos < end):
        start = max(0, min(pos, n - max_len))
        end = start + max_len
    return start, end


class BertMaskedItemDataset(Dataset):
    """Single-mask BERT4Rec pretraining dataset.

    Each row is a right-aligned sequence with one item replaced by [MASK].
    The target is the original masked item. We use in-batch sampled softmax/InfoNCE
    instead of full softmax over ~1M items for practical local training.
    """
    def __init__(self, seq_user_ids: np.ndarray, seqs: list[np.ndarray], event_seqs: list[np.ndarray], recency_seqs: list[np.ndarray], examples: np.ndarray, max_len: int, mask_idx: int):
        self.seq_user_ids = np.asarray(seq_user_ids, dtype=np.int64)
        self.seqs = seqs
        self.event_seqs = event_seqs
        self.recency_seqs = recency_seqs
        self.examples = np.asarray(examples, dtype=np.int32)
        self.max_len = int(max_len)
        self.mask_idx = int(mask_idx)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int):
        uidx, pos = self.examples[idx]
        uidx = int(uidx)
        pos = int(pos)
        seq = self.seqs[uidx]
        ev = self.event_seqs[uidx]
        rb = self.recency_seqs[uidx]
        start, end = _bert_window_around_position(seq, pos, self.max_len)
        window = np.asarray(seq[start:end], dtype=np.int64).copy()
        window_ev = np.asarray(ev[start:end], dtype=np.int64).copy()
        window_rb = np.asarray(rb[start:end], dtype=np.int64).copy()
        rel_pos = pos - start
        target = int(window[rel_pos])

        r = random.random()
        if r < CFG.mask_random_item_prob:
            window[rel_pos] = random.randint(1, self.mask_idx - 1)
        elif r < CFG.mask_random_item_prob + CFG.mask_keep_original_prob:
            pass
        else:
            window[rel_pos] = self.mask_idx
            window_ev[rel_pos] = 0
            window_rb[rel_pos] = 0

        user_id = int(self.seq_user_ids[uidx])
        return (
            torch.tensor(uidx, dtype=torch.long),
            torch.tensor(user_id, dtype=torch.long),
            torch.from_numpy(_pad_right_aligned(window, self.max_len, np.int64)),
            torch.from_numpy(_pad_right_aligned(window_ev, self.max_len, np.int64)),
            torch.from_numpy(_pad_right_aligned(window_rb, self.max_len, np.int64)),
            torch.tensor(target, dtype=torch.long),
        )


NextItemDataset = BertMaskedItemDataset


class OrderTargetDataset(Dataset):
    """Future-order fine-tune dataset.

    Input is context sequence with a [MASK] token appended at the end.
    Target is a future ordered item from the target window.
    """
    def __init__(self, user_ids: np.ndarray, user_seq: dict[int, np.ndarray], user_event_seq: dict[int, np.ndarray], user_recency_seq: dict[int, np.ndarray], pairs: np.ndarray, max_len: int, mask_idx: int):
        self.user_ids = np.asarray(user_ids, dtype=np.int64)
        self.user_seq = user_seq
        self.user_event_seq = user_event_seq
        self.user_recency_seq = user_recency_seq
        self.pairs = np.asarray(pairs, dtype=np.int64)
        self.max_len = int(max_len)
        self.mask_idx = int(mask_idx)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx: int):
        user_id = int(self.pairs[idx, 0])
        target = int(self.pairs[idx, 1])
        seq = self.user_seq.get(user_id, np.empty(0, dtype=np.int32))
        ev = self.user_event_seq.get(user_id, np.empty(0, dtype=np.int16))
        rb = self.user_recency_seq.get(user_id, np.empty(0, dtype=np.int16))

        seq = np.asarray(seq[-(self.max_len - 1):], dtype=np.int64)
        ev = np.asarray(ev[-(self.max_len - 1):], dtype=np.int64)
        rb = np.asarray(rb[-(self.max_len - 1):], dtype=np.int64)
        seq = np.concatenate([seq, np.asarray([self.mask_idx], dtype=np.int64)])
        ev = np.concatenate([ev, np.asarray([0], dtype=np.int64)])
        rb = np.concatenate([rb, np.asarray([0], dtype=np.int64)])

        return (
            torch.tensor(0, dtype=torch.long),
            torch.tensor(user_id, dtype=torch.long),
            torch.from_numpy(_pad_right_aligned(seq, self.max_len, np.int64)),
            torch.from_numpy(_pad_right_aligned(ev, self.max_len, np.int64)),
            torch.from_numpy(_pad_right_aligned(rb, self.max_len, np.int64)),
            torch.tensor(target, dtype=torch.long),
        )


def build_order_target_pairs(context_split: str, item2idx: dict[int, int]) -> tuple[np.ndarray, np.ndarray, dict[int, np.ndarray], dict[int, np.ndarray], dict[int, np.ndarray]]:
    """Future-order pairs for order-focused fine-tune. Uses ranker_train by default to avoid validation leakage."""
    user_ids, user_seq, user_event_seq, user_recency_seq = build_sequences_with_checkpoint_vocab(context_split, item2idx)
    _, _, target_start, target_end = split_spec(context_split)

    item_map = pl.DataFrame({
        "item_id": np.fromiter(item2idx.keys(), dtype=np.int64),
        "item_idx": np.fromiter(item2idx.values(), dtype=np.int32),
    }).lazy()
    users_lf = pl.DataFrame({"user_id": user_ids}).lazy()

    lf = (
        read_orders_lf("user_item", target_start, target_end)
        .join(users_lf, on="user_id", how="inner")
        .join(item_map, on="item_id", how="inner")
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_idx").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_idx"])
    )
    df = collect_streaming(lf)
    if df.is_empty():
        raise RuntimeError(f"No order fine-tune targets for split={context_split}")
    pairs = np.stack([df["user_id"].to_numpy().astype(np.int64), df["item_idx"].to_numpy().astype(np.int64)], axis=1)
    rng = np.random.default_rng(CFG.random_state)
    rng.shuffle(pairs)
    print(f"order fine-tune pairs from {context_split}: {len(pairs):,}")
    return pairs, user_ids, user_seq, user_event_seq, user_recency_seq

# -----------------------------
# BERT4Rec model
# -----------------------------

class BERT4RecCandidateGenerator(nn.Module):
    """BERT4Rec-style bidirectional Transformer for retrieval.

    Training:
    - pretrain: one observed item in the context is replaced by [MASK], target is the original item;
    - fine-tune: [MASK] is appended after context, target is a future order item.

    Retrieval:
    - encode the final [MASK] token as user/query vector;
    - ANN search over item tower vectors.
    """
    def __init__(
        self,
        *,
        num_items: int,
        max_len: int,
        emb_dim: int,
        num_heads: int,
        num_layers: int,
        dropout: float,
        catalog_idx_by_item_idx: np.ndarray | None = None,
        cat_l0_idx_by_item_idx: np.ndarray | None = None,
        num_catalogs: int = 1,
        num_cat_l0: int = 1,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.mask_idx = int(num_items)
        self.num_tokens = int(num_items + 1)
        self.max_len = int(max_len)
        self.emb_dim = int(emb_dim)
        self.use_catalog = bool(CFG.use_catalog_embeddings and catalog_idx_by_item_idx is not None and cat_l0_idx_by_item_idx is not None)
        self.use_event_type = bool(CFG.use_event_type_embeddings)
        self.use_recency = bool(CFG.use_recency_embeddings)

        self.item_embedding = nn.Embedding(self.num_tokens, emb_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, emb_dim)

        if self.use_catalog:
            cat_arr = np.zeros(self.num_tokens, dtype=np.int64)
            l0_arr = np.zeros(self.num_tokens, dtype=np.int64)
            base_cat = np.asarray(catalog_idx_by_item_idx, dtype=np.int64)
            base_l0 = np.asarray(cat_l0_idx_by_item_idx, dtype=np.int64)
            cat_arr[:len(base_cat)] = base_cat
            l0_arr[:len(base_l0)] = base_l0
            self.catalog_embedding = nn.Embedding(max(1, num_catalogs), CFG.catalog_dim, padding_idx=0)
            self.cat_l0_embedding = nn.Embedding(max(1, num_cat_l0), CFG.cat_l0_dim, padding_idx=0)
            self.side_project = nn.Linear(CFG.catalog_dim + CFG.cat_l0_dim, emb_dim)
            self.register_buffer("catalog_idx_by_item_idx", torch.as_tensor(cat_arr, dtype=torch.long), persistent=False)
            self.register_buffer("cat_l0_idx_by_item_idx", torch.as_tensor(l0_arr, dtype=torch.long), persistent=False)
        else:
            self.catalog_embedding = None
            self.cat_l0_embedding = None
            self.side_project = None
            self.register_buffer("catalog_idx_by_item_idx", torch.zeros(self.num_tokens, dtype=torch.long), persistent=False)
            self.register_buffer("cat_l0_idx_by_item_idx", torch.zeros(self.num_tokens, dtype=torch.long), persistent=False)

        if self.use_event_type:
            self.event_type_embedding = nn.Embedding(max(2, CFG.num_event_types), CFG.event_type_dim, padding_idx=0)
            self.event_type_project = nn.Linear(CFG.event_type_dim, emb_dim)
        else:
            self.event_type_embedding = None
            self.event_type_project = None

        if self.use_recency:
            self.num_recency_buckets = len(CFG.recency_bucket_edges_days) + 2
            self.recency_embedding = nn.Embedding(self.num_recency_buckets, CFG.recency_dim, padding_idx=0)
            self.recency_project = nn.Linear(CFG.recency_dim, emb_dim)
        else:
            self.num_recency_buckets = 1
            self.recency_embedding = None
            self.recency_project = None

        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.temperature = nn.Parameter(torch.tensor(float(CFG.temperature_init)))

    def item_side_vector(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_tokens - 1)
        base = self.item_embedding(item_idx)
        if not self.use_catalog:
            return base
        safe_idx = item_idx.clamp(0, self.catalog_idx_by_item_idx.numel() - 1)
        catalog_idx = self.catalog_idx_by_item_idx[safe_idx]
        cat_l0_idx = self.cat_l0_idx_by_item_idx[safe_idx]
        side = torch.cat([self.catalog_embedding(catalog_idx), self.cat_l0_embedding(cat_l0_idx)], dim=-1)
        return base + self.side_project(side)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.item_side_vector(item_idx), dim=-1)

    def encode_user(self, user_idx: torch.Tensor, hist: torch.Tensor, event_type: torch.Tensor | None = None, recency_bucket: torch.Tensor | None = None) -> torch.Tensor:
        bsz, seq_len = hist.shape
        pos = torch.arange(seq_len, device=hist.device).unsqueeze(0).expand(bsz, -1)
        x = self.item_side_vector(hist) + self.pos_embedding(pos)

        if self.use_event_type:
            if event_type is None:
                event_type = torch.zeros_like(hist)
            event_type = event_type.clamp(0, CFG.num_event_types - 1)
            x = x + self.event_type_project(self.event_type_embedding(event_type))

        if self.use_recency:
            if recency_bucket is None:
                recency_bucket = torch.zeros_like(hist)
            recency_bucket = recency_bucket.clamp(0, self.num_recency_buckets - 1)
            x = x + self.recency_project(self.recency_embedding(recency_bucket))

        x = self.dropout(x)
        padding_mask = hist.eq(0)
        h = self.encoder(x, src_key_padding_mask=padding_mask)
        h = self.norm(h)

        is_mask = hist.eq(self.mask_idx)
        has_mask = is_mask.any(dim=1)
        positions = torch.arange(seq_len, device=hist.device).view(1, -1).expand(bsz, -1)
        mask_pos = torch.where(is_mask, positions, torch.zeros_like(positions)).max(dim=1).values
        nonpad = hist.ne(0)
        last_nonpad = torch.where(nonpad, positions, torch.zeros_like(positions)).max(dim=1).values
        take_pos = torch.where(has_mask, mask_pos, last_nonpad)
        z = h[torch.arange(bsz, device=hist.device), take_pos, :]
        empty = hist.ne(0).sum(dim=1).eq(0)
        if empty.any():
            z = z.masked_fill(empty.unsqueeze(1), 0.0)
        return F.normalize(z, dim=-1)

    def logits(self, user_idx: torch.Tensor, hist: torch.Tensor, event_type: torch.Tensor, recency_bucket: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.encode_user(user_idx, hist, event_type, recency_bucket)
        v = self.encode_item_idx(item_idx)
        temp = self.temperature.clamp(0.02, 0.20)
        return (u @ v.T) / temp

# -----------------------------
# Loss and training
# -----------------------------

def make_grad_scaler():
    if CFG.use_amp and torch.cuda.is_available():
        try:
            return torch.amp.GradScaler("cuda")
        except Exception:
            return torch.cuda.amp.GradScaler()
    return torch.amp.GradScaler(enabled=False)


def autocast_context():
    if CFG.use_amp and torch.cuda.is_available():
        try:
            return torch.amp.autocast("cuda", dtype=torch.float16)
        except Exception:
            return torch.cuda.amp.autocast(dtype=torch.float16)
    from contextlib import nullcontext
    return nullcontext()


def item_log_q_from_weights(item_weight_by_idx: np.ndarray) -> np.ndarray:
    w = np.asarray(item_weight_by_idx, dtype=np.float64)
    w = np.maximum(w, 0.0)
    if len(w) > 0:
        w[0] = 0.0
    q = w + float(CFG.logq_smoothing)
    if len(q) > 0:
        q[0] = float(CFG.logq_smoothing)
    q = q / max(q.sum(), 1e-12)
    return np.log(np.maximum(q, 1e-12)).astype(np.float32)


def attach_item_log_q(model: nn.Module, item_weight_by_idx: np.ndarray) -> None:
    log_q = item_log_q_from_weights(item_weight_by_idx)
    model.register_buffer("item_log_q_by_idx", torch.from_numpy(log_q), persistent=False)


def multipositive_inbatch_loss(
    model: BERT4RecCandidateGenerator,
    user_idx: torch.Tensor,
    user_id: torch.Tensor,
    hist: torch.Tensor,
    event_type: torch.Tensor,
    recency_bucket: torch.Tensor,
    pos: torch.Tensor,
) -> tuple[torch.Tensor, dict]:
    logits = model.logits(user_idx, hist, event_type, recency_bucket, pos)

    if hasattr(model, "item_log_q_by_idx") and CFG.popularity_correction_beta > 0:
        log_q = model.item_log_q_by_idx[pos].to(logits.device).view(1, -1)
        logits = logits - float(CFG.popularity_correction_beta) * log_q

    same_item = pos.view(-1, 1).eq(pos.view(1, -1))
    if CFG.multipositive_same_user:
        same_user = user_id.view(-1, 1).eq(user_id.view(1, -1))
        pos_mask = same_item | same_user
    else:
        pos_mask = same_item
    pos_mask = pos_mask.float()

    log_probs = F.log_softmax(logits, dim=1)
    denom = pos_mask.sum(dim=1).clamp_min(1.0)
    loss = -(log_probs * pos_mask).sum(dim=1) / denom
    return loss.mean(), {
        "loss": float(loss.mean().detach().cpu()),
        "temp": float(model.temperature.detach().clamp(0.02, 0.20).cpu()),
    }


@torch.no_grad()
def evaluate_loss(model: nn.Module, dataset: Dataset, batch_size: int) -> dict:
    if len(dataset) == 0:
        return {}
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=False,
    )
    model.eval()
    vals = []
    for user_idx, user_id, hist, event_type, recency_bucket, pos in tqdm(loader, desc="Valid loss", leave=False):
        user_idx = user_idx.to(DEVICE, non_blocking=True)
        user_id = user_id.to(DEVICE, non_blocking=True)
        hist = hist.to(DEVICE, non_blocking=True)
        event_type = event_type.to(DEVICE, non_blocking=True)
        recency_bucket = recency_bucket.to(DEVICE, non_blocking=True)
        pos = pos.to(DEVICE, non_blocking=True)
        with autocast_context():
            loss, _ = multipositive_inbatch_loss(model, user_idx, user_id, hist, event_type, recency_bucket, pos)
        vals.append(float(loss.detach().cpu()))
    return {"valid_loss": float(np.mean(vals)) if vals else np.nan}


def train_bert4rec_stage(
    model: nn.Module,
    train_dataset: Dataset,
    valid_dataset: Dataset,
    *,
    epochs: int,
    batch_size: int,
    lr: float,
    stage_name: str,
) -> dict:
    """Train one stage.

    v2 change vs fast version:
    - pretrain may use OneCycleLR;
    - future-order fine-tune uses constant LR by default, so lr is not annealed to ~1e-7;
    - scheduler.step() is called only after optimizer.step();
    - best checkpoint is selected by validation loss when enabled.
    """
    model = model.to(DEVICE)

    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=True,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.weight_decay)
    total_steps = max(1, epochs * len(loader))

    is_finetune = "finetune" in stage_name.lower()
    if is_finetune:
        scheduler = None
        scheduler_name = "constant_lr"
    else:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=total_steps,
            pct_start=CFG.scheduler_pct_start,
            anneal_strategy="cos",
            div_factor=10.0,
            final_div_factor=20.0,
        )
        scheduler_name = "onecycle_lr"

    scaler = make_grad_scaler()

    best_state = None
    best_loss = float("inf")
    history = []
    t0 = time.perf_counter()
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()
        rolling = defaultdict(list)
        pbar = tqdm(loader, desc=f"{stage_name} epoch {epoch}/{epochs}")
        for user_idx, user_id, hist, event_type, recency_bucket, pos in pbar:
            user_idx = user_idx.to(DEVICE, non_blocking=True)
            user_id = user_id.to(DEVICE, non_blocking=True)
            hist = hist.to(DEVICE, non_blocking=True)
            event_type = event_type.to(DEVICE, non_blocking=True)
            recency_bucket = recency_bucket.to(DEVICE, non_blocking=True)
            pos = pos.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                loss, metrics = multipositive_inbatch_loss(model, user_idx, user_id, hist, event_type, recency_bucket, pos)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None:
                scheduler.step()

            global_step += 1
            for k, v in metrics.items():
                rolling[k].append(v)
            if global_step % CFG.log_every_steps == 0:
                pbar.set_postfix(
                    loss=np.mean(rolling["loss"][-CFG.log_every_steps:]),
                    temp=np.mean(rolling["temp"][-CFG.log_every_steps:]),
                    lr=optimizer.param_groups[0]["lr"],
                    scheduler=scheduler_name,
                )

        row = {
            "stage": stage_name,
            "epoch": epoch,
            "scheduler": scheduler_name,
            **{k: float(np.mean(v)) for k, v in rolling.items() if v},
            "lr": float(optimizer.param_groups[0]["lr"]),
        }
        if CFG.eval_every_epoch and len(valid_dataset) > 0:
            row.update(evaluate_loss(model, valid_dataset, batch_size=batch_size))
            current = row.get("valid_loss", row.get("loss", float("inf")))
        else:
            current = row.get("loss", float("inf"))

        if current < best_loss:
            best_loss = current
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history.append(row)
        print(row)

    if best_state is not None:
        model.load_state_dict(best_state, strict=False)

    return {"stage": stage_name, "train_time_sec": float(time.perf_counter() - t0), "best_loss": float(best_loss), "history": history}

def init_bert4rec_model(item_id_by_idx: np.ndarray, item_weight_by_idx: np.ndarray, side: dict) -> nn.Module:
    model = BERT4RecCandidateGenerator(
        num_items=len(item_id_by_idx),
        max_len=CFG.sequence_max_len,
        emb_dim=CFG.embedding_dim,
        num_heads=CFG.num_heads,
        num_layers=CFG.num_layers,
        dropout=CFG.dropout,
        catalog_idx_by_item_idx=side.get("catalog_idx_by_item_idx"),
        cat_l0_idx_by_item_idx=side.get("cat_l0_idx_by_item_idx"),
        num_catalogs=side.get("num_catalogs", 1),
        num_cat_l0=side.get("num_cat_l0", 1),
    )
    attach_item_log_q(model, item_weight_by_idx)
    return model


def checkpoint_dir(split: str | None = None) -> Path:
    s = split or CFG.checkpoint_split
    return MODELS_DIR / CFG.source_name / s


def checkpoint_path(split: str | None = None) -> Path:
    return checkpoint_dir(split) / "model.pt"


def prepare_training_data(split_name: str):
    item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, event_seqs, recency_seqs, user_seq, user_event_seq, user_recency_seq, meta = load_or_build_training_sequences(split_name)
    examples = make_next_item_examples(
        seqs,
        max_examples=CFG.max_train_examples,
        max_targets_per_user=CFG.max_targets_per_user,
        seed=CFG.random_state,
    )
    train_examples, valid_examples = split_train_valid_examples(examples, CFG.random_state)
    train_ds = NextItemDataset(seq_user_ids, seqs, event_seqs, recency_seqs, train_examples, CFG.sequence_max_len, mask_idx=len(item_id_by_idx))
    valid_ds = NextItemDataset(seq_user_ids, seqs, event_seqs, recency_seqs, valid_examples, CFG.sequence_max_len, mask_idx=len(item_id_by_idx))
    side = build_item_side_arrays(item_id_by_idx, PROCESSED_DIR / "item_side", split_name)
    return {
        "item2idx": item2idx,
        "item_id_by_idx": item_id_by_idx,
        "item_weight_by_idx": item_weight_by_idx,
        "seq_user_ids": seq_user_ids,
        "seqs": seqs,
        "event_seqs": event_seqs,
        "recency_seqs": recency_seqs,
        "user_seq": user_seq,
        "user_event_seq": user_event_seq,
        "user_recency_seq": user_recency_seq,
        "meta": meta,
        "train_ds": train_ds,
        "valid_ds": valid_ds,
        "side": side,
    }


def train_or_load_bert4rec(split_name: str | None = None) -> tuple[nn.Module, dict]:
    split_name = split_name or CFG.checkpoint_split
    model_path = checkpoint_path(split_name)
    meta_path = checkpoint_dir(split_name) / "run_meta.json"

    data = prepare_training_data(split_name)
    should_train = CFG.force_retrain_bert4rec or CFG.rebuild_model or (CFG.train_bert4rec_if_missing and not model_path.exists())

    if model_path.exists() and not should_train:
        print(f"BERT4Rec checkpoint exists, loading: {rel(model_path)}")
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        model = init_bert4rec_model(data["item_id_by_idx"], data["item_weight_by_idx"], data["side"])
        load = model.load_state_dict(ckpt["model_state_dict"], strict=False)
        if load.missing_keys or load.unexpected_keys:
            print("load_state_dict:", "missing", load.missing_keys[:5], "unexpected", load.unexpected_keys[:5])
        return model, {**data, "train_info": ckpt.get("train_info", {})}

    print(f"Training BERT4Rec checkpoint for split={split_name}")
    model = init_bert4rec_model(data["item_id_by_idx"], data["item_weight_by_idx"], data["side"])

    train_info = {"stages": []}
    pre = train_bert4rec_stage(
        model,
        data["train_ds"],
        data["valid_ds"],
        epochs=CFG.pretrain_epochs,
        batch_size=CFG.batch_size,
        lr=CFG.lr,
        stage_name=f"{CFG.source_name}/{split_name}/pretrain_next_item",
    )
    train_info["stages"].append(pre)

    if CFG.finetune_on_orders:
        try:
            pairs, ft_user_ids, ft_user_seq, ft_user_event_seq, ft_user_recency_seq = build_order_target_pairs("ranker_train", data["item2idx"])
            # Split fine-tune pairs.
            rng = np.random.default_rng(CFG.random_state)
            idx = np.arange(len(pairs))
            rng.shuffle(idx)
            n_valid = min(CFG.max_valid_examples, max(1, int(len(pairs) * CFG.train_valid_fraction)))
            valid_pairs = pairs[idx[:n_valid]]
            train_pairs = pairs[idx[n_valid:]]
            train_ft = OrderTargetDataset(ft_user_ids, ft_user_seq, ft_user_event_seq, ft_user_recency_seq, train_pairs, CFG.sequence_max_len, mask_idx=len(data["item_id_by_idx"]))
            valid_ft = OrderTargetDataset(ft_user_ids, ft_user_seq, ft_user_event_seq, ft_user_recency_seq, valid_pairs, CFG.sequence_max_len, mask_idx=len(data["item_id_by_idx"]))
            ft = train_bert4rec_stage(
                model,
                train_ft,
                valid_ft,
                epochs=CFG.finetune_epochs,
                batch_size=CFG.finetune_batch_size,
                lr=CFG.finetune_lr,
                stage_name=f"{CFG.source_name}/{split_name}/finetune_future_orders",
            )
            train_info["stages"].append(ft)
        except Exception as e:
            warnings.warn(f"Order fine-tune skipped: {type(e).__name__}: {e}")

    ensure_dir(checkpoint_dir(split_name))
    model_cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save({
        "model_state_dict": model_cpu_state,
        "source_name": CFG.source_name,
        "model_kind": "bert4rec",
        "item_id_by_idx": data["item_id_by_idx"],
        "item_weight_by_idx": data["item_weight_by_idx"],
        "config": asdict(CFG),
        "train_info": train_info,
        "hparams": {
            "embedding_dim": CFG.embedding_dim,
            "num_heads": CFG.num_heads,
            "num_layers": CFG.num_layers,
            "dropout": CFG.dropout,
            "use_catalog_embeddings": CFG.use_catalog_embeddings,
            "use_event_type_embeddings": CFG.use_event_type_embeddings,
            "use_recency_embeddings": CFG.use_recency_embeddings,
            "pretrain_epochs": CFG.pretrain_epochs,
            "finetune_on_orders": CFG.finetune_on_orders,
            "finetune_epochs": CFG.finetune_epochs,
            "popularity_correction_beta": CFG.popularity_correction_beta,
        },
    }, model_path)
    meta_path.write_text(json.dumps({
        "source_name": CFG.source_name,
        "split": split_name,
        "model_path": rel(model_path),
        "trained_at_utc": datetime.now(timezone.utc).isoformat(),
        "train_info": train_info,
        "config": asdict(CFG),
    }, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"BERT4Rec saved: {rel(model_path)}")

    return model, {**data, "train_info": train_info}

In [7]:
# -----------------------------
# ANN retrieval and candidate generation
# -----------------------------

@torch.no_grad()
def compute_item_vectors(model: nn.Module, num_items: int, batch_size: int = 65536) -> np.ndarray:
    model = model.to(DEVICE)
    model.eval()
    chunks = []
    for start in trange(1, num_items, batch_size, desc="Encode BERT4Rec items"):
        end = min(num_items, start + batch_size)
        idx = torch.arange(start, end, device=DEVICE, dtype=torch.long)
        vec = model.encode_item_idx(idx).detach().float().cpu().numpy()
        chunks.append(vec.astype(np.float32))
    return np.vstack(chunks).astype(np.float32) if chunks else np.empty((0, CFG.embedding_dim), dtype=np.float32)


@torch.no_grad()
@torch.no_grad()
def encode_user_vectors(model: nn.Module, hists: np.ndarray, event_types: np.ndarray, recency_buckets: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    model = model.to(DEVICE)
    model.eval()
    out = []
    for start in range(0, len(hists), batch_size):
        end = min(len(hists), start + batch_size)
        user_idx = torch.zeros(end - start, dtype=torch.long, device=DEVICE)
        hist_np = hists[start:end].astype(np.int64).copy()
        ev_np = event_types[start:end].astype(np.int64).copy()
        rb_np = recency_buckets[start:end].astype(np.int64).copy()
        # Inference query: keep last max_len-1 observed events and append [MASK].
        if hist_np.shape[1] >= 1:
            hist_np[:, :-1] = hist_np[:, 1:]
            ev_np[:, :-1] = ev_np[:, 1:]
            rb_np[:, :-1] = rb_np[:, 1:]
            hist_np[:, -1] = model.mask_idx
            ev_np[:, -1] = 0
            rb_np[:, -1] = 0
        hist = torch.from_numpy(hist_np).to(DEVICE)
        ev = torch.from_numpy(ev_np).to(DEVICE)
        rb = torch.from_numpy(rb_np).to(DEVICE)
        vec = model.encode_user(user_idx, hist, ev, rb).detach().float().cpu().numpy()
        out.append(vec.astype(np.float32))
    return np.vstack(out).astype(np.float32) if out else np.empty((0, CFG.embedding_dim), dtype=np.float32)


@dataclass
class ANNIndex:
    backend: str
    index: object | None
    item_vecs: np.ndarray | None

    def query(self, user_vecs: np.ndarray, topk: int) -> tuple[np.ndarray, np.ndarray]:
        topk = int(min(topk, self.item_vecs.shape[0] if self.item_vecs is not None else topk))
        user_vecs = np.asarray(user_vecs, dtype=np.float32)
        if self.backend == "hnswlib":
            labels, dist = self.index.knn_query(user_vecs, k=topk)
            return labels.astype(np.int64), (1.0 - dist).astype(np.float32)
        if self.backend == "faiss":
            scores, labels = self.index.search(np.ascontiguousarray(user_vecs.astype(np.float32)), topk)
            return labels.astype(np.int64), scores.astype(np.float32)
        if self.backend == "torch":
            return exact_topk_torch(user_vecs, self.item_vecs, topk)
        if self.backend == "numpy":
            return topk_numpy(user_vecs, self.item_vecs, topk)
        raise ValueError(self.backend)


def resolve_ann_backend() -> str:
    if CFG.ann_backend != "auto":
        return CFG.ann_backend
    if HAS_HNSWLIB:
        return "hnswlib"
    if HAS_FAISS:
        return "faiss"
    if torch.cuda.is_available():
        return "torch"
    return "numpy"


def build_or_load_ann(model: nn.Module, item_id_by_idx: np.ndarray) -> ANNIndex:
    ann_dir = checkpoint_dir(CFG.checkpoint_split)
    ensure_dir(ann_dir)
    backend = resolve_ann_backend()
    n_items = len(item_id_by_idx) - 1

    if backend == "hnswlib":
        if not HAS_HNSWLIB:
            raise ImportError("hnswlib is not installed")
        index_path = ann_dir / f"{CFG.source_name}_{CFG.checkpoint_split}_hnsw.bin"
        if index_path.exists() and not CFG.rebuild_ann_index:
            idx = hnswlib.Index(space="cosine", dim=CFG.embedding_dim)
            idx.load_index(str(index_path), max_elements=n_items)
            idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
            print(f"HNSW loaded: {rel(index_path)}")
            # item_vecs only needed for topk limit; store dummy shape.
            dummy = np.empty((n_items, CFG.embedding_dim), dtype=np.float32)
            return ANNIndex("hnswlib", idx, dummy)

        item_vecs = compute_item_vectors(model, len(item_id_by_idx))
        idx = hnswlib.Index(space="cosine", dim=item_vecs.shape[1])
        idx.init_index(max_elements=item_vecs.shape[0], ef_construction=CFG.hnsw_ef_construction, M=CFG.hnsw_m)
        for s in trange(0, item_vecs.shape[0], CFG.hnsw_add_batch_size, desc="Build BERT4Rec HNSW"):
            e = min(item_vecs.shape[0], s + CFG.hnsw_add_batch_size)
            idx.add_items(item_vecs[s:e], np.arange(s, e, dtype=np.int64), num_threads=CFG.hnsw_num_threads)
        idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
        idx.save_index(str(index_path))
        print(f"HNSW saved: {rel(index_path)}")
        return ANNIndex("hnswlib", idx, item_vecs)

    item_vecs = compute_item_vectors(model, len(item_id_by_idx))
    if backend == "faiss":
        if not HAS_FAISS:
            raise ImportError("faiss is not installed")
        xb = np.ascontiguousarray(item_vecs.astype(np.float32))
        faiss.normalize_L2(xb)
        index_path = ann_dir / f"{CFG.source_name}_{CFG.checkpoint_split}_faiss_{CFG.faiss_index_type}.index"
        if index_path.exists() and not CFG.rebuild_ann_index:
            idx = faiss.read_index(str(index_path))
            print(f"FAISS loaded: {rel(index_path)}")
        else:
            if CFG.faiss_index_type == "flat" or xb.shape[0] < max(10_000, CFG.faiss_nlist * 20):
                idx = faiss.IndexFlatIP(xb.shape[1])
            else:
                nlist = int(min(CFG.faiss_nlist, max(1, math.sqrt(xb.shape[0]) * 4)))
                quantizer = faiss.IndexFlatIP(xb.shape[1])
                idx = faiss.IndexIVFFlat(quantizer, xb.shape[1], nlist, faiss.METRIC_INNER_PRODUCT)
                sample_n = min(xb.shape[0], CFG.faiss_train_sample_size)
                rng = np.random.default_rng(CFG.random_state)
                train_idx = rng.choice(xb.shape[0], size=sample_n, replace=False) if sample_n < xb.shape[0] else np.arange(xb.shape[0])
                idx.train(np.ascontiguousarray(xb[train_idx]))
                idx.nprobe = int(min(CFG.faiss_nprobe, nlist))
            idx.add(xb)
            faiss.write_index(idx, str(index_path))
            print(f"FAISS saved: {rel(index_path)}")
        if CFG.faiss_use_gpu and hasattr(faiss, "get_num_gpus") and faiss.get_num_gpus() > 0:
            res = faiss.StandardGpuResources()
            idx = faiss.index_cpu_to_gpu(res, 0, idx)
        if hasattr(idx, "nprobe"):
            idx.nprobe = int(min(CFG.faiss_nprobe, getattr(idx, "nlist", CFG.faiss_nprobe)))
        return ANNIndex("faiss", idx, xb)

    if backend == "torch":
        return ANNIndex("torch", None, item_vecs)
    if backend == "numpy":
        return ANNIndex("numpy", None, item_vecs)
    raise ValueError(backend)


def exact_topk_torch(user_vecs: np.ndarray, item_vecs: np.ndarray, topk: int, batch_size: int = 512) -> tuple[np.ndarray, np.ndarray]:
    dev = DEVICE if torch.cuda.is_available() else torch.device("cpu")
    item_t = torch.from_numpy(item_vecs).to(dev).float()
    item_t = F.normalize(item_t, dim=1)
    all_idx, all_scores = [], []
    for start in trange(0, len(user_vecs), batch_size, desc="Torch exact retrieval"):
        u = torch.from_numpy(user_vecs[start:start + batch_size]).to(dev).float()
        u = F.normalize(u, dim=1)
        scores = u @ item_t.T
        vals, idx = torch.topk(scores, k=min(topk, item_t.shape[0]), dim=1)
        all_idx.append(idx.detach().cpu().numpy().astype(np.int64))
        all_scores.append(vals.detach().cpu().numpy().astype(np.float32))
    return np.vstack(all_idx), np.vstack(all_scores)


def topk_numpy(user_vecs: np.ndarray, item_vecs: np.ndarray, topk: int, batch_size: int = 256) -> tuple[np.ndarray, np.ndarray]:
    item = item_vecs / np.maximum(np.linalg.norm(item_vecs, axis=1, keepdims=True), 1e-12)
    out_idx, out_scores = [], []
    for start in trange(0, len(user_vecs), batch_size, desc="Numpy retrieval"):
        u = user_vecs[start:start + batch_size]
        u = u / np.maximum(np.linalg.norm(u, axis=1, keepdims=True), 1e-12)
        scores = u @ item.T
        kk = min(topk, scores.shape[1])
        idx = np.argpartition(-scores, kk - 1, axis=1)[:, :kk]
        vals = np.take_along_axis(scores, idx, axis=1)
        order = np.argsort(-vals, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        vals = np.take_along_axis(vals, order, axis=1)
        out_idx.append(idx.astype(np.int64))
        out_scores.append(vals.astype(np.float32))
    return np.vstack(out_idx), np.vstack(out_scores)


def write_candidate_buffer(rows: list[tuple], out_dir: Path, part_id: int) -> int:
    if not rows:
        return 0
    source = CFG.source_name
    df = pd.DataFrame(rows, columns=["user_id", "item_id", "score", "rank"])
    df["source"] = source
    df[f"{source}_score"] = df["score"].astype("float32")
    df[f"{source}_rank"] = df["rank"].astype("int32")
    df[f"in_{source}"] = np.int8(1)
    df[f"f_num__candidate__in_{source}"] = np.float32(1.0)
    df[f"f_num__candidate__{source}_score"] = df["score"].astype("float32")
    df[f"f_num__candidate__{source}_rank"] = df["rank"].astype("float32")
    df[f"f_num__candidate__{source}_rr"] = (1.0 / df["rank"].astype("float32")).astype("float32")
    df["user_shard"] = (df["user_id"].astype("int64") % CFG.n_shards).astype("int16")

    written = 0
    for shard_id, part in df.groupby("user_shard", sort=True):
        sdir = out_dir / f"user_shard={int(shard_id)}"
        sdir.mkdir(parents=True, exist_ok=True)
        out_file = sdir / f"{source}_part_{part_id:05d}.parquet"
        pl.from_pandas(part.drop(columns=["user_shard"])).write_parquet(out_file, compression="zstd")
        written += len(part)
    return written


def generate_bert4rec_candidates_for_split(
    model: nn.Module,
    ann: ANNIndex,
    split_name: str,
    item2idx: dict[int, int],
    item_id_by_idx: np.ndarray,
    item_weight_by_idx: np.ndarray,
) -> dict:
    out_dir = CANDIDATES_DIR / CFG.source_name / split_name
    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_sources:
        print(f"{CFG.source_name}/{split_name}: reused -> {rel(out_dir)}")
        meta = json.loads((out_dir / "_meta.json").read_text(encoding="utf-8"))
        return {"split": split_name, "status": "reused", "rows": meta.get("rows")}

    clean_output_dir(out_dir)
    user_ids, user_seq, user_event_seq, user_recency_seq = build_sequences_with_checkpoint_vocab(split_name, item2idx)
    t0 = time.perf_counter()
    # Popularity prior is computed over item_idx. ANN row id 0 corresponds to item_idx=1.
    item_pop = np.log1p(np.asarray(item_weight_by_idx, dtype=np.float32))
    if item_pop.size > 1:
        mu = float(np.nanmean(item_pop[1:]))
        sigma = float(np.nanstd(item_pop[1:]) + 1e-6)
        item_pop = ((item_pop - mu) / sigma).astype(np.float32)
    else:
        item_pop = np.zeros_like(item_pop, dtype=np.float32)

    # Category/catalog arrays for post-ANN calibration.
    if hasattr(model, "catalog_idx_by_item_idx"):
        catalog_arr = model.catalog_idx_by_item_idx.detach().cpu().numpy().astype(np.int64)
        cat_l0_arr = model.cat_l0_idx_by_item_idx.detach().cpu().numpy().astype(np.int64)
    else:
        catalog_arr = np.zeros(len(item_id_by_idx), dtype=np.int64)
        cat_l0_arr = np.zeros(len(item_id_by_idx), dtype=np.int64)
    total_rows = 0
    rows_buffer: list[tuple] = []
    part_id = 0

    for start in trange(0, len(user_ids), CFG.retrieval_user_batch_size, desc=f"Generate {CFG.source_name}/{split_name}"):
        users = user_ids[start:start + CFG.retrieval_user_batch_size]
        hists = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        event_types = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        recency_buckets = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        keep = []
        for j, user_id in enumerate(users):
            seq = user_seq.get(int(user_id))
            if seq is None or len(seq) == 0:
                continue
            ev = user_event_seq.get(int(user_id), np.ones(len(seq), dtype=np.int16))
            rb = user_recency_seq.get(int(user_id), np.ones(len(seq), dtype=np.int16))
            hist = seq[-CFG.sequence_max_len:]
            hist_ev = ev[-CFG.sequence_max_len:]
            hist_rb = rb[-CFG.sequence_max_len:]
            hists[j, -len(hist):] = hist.astype(np.int64)
            event_types[j, -len(hist_ev):] = hist_ev.astype(np.int64)
            recency_buckets[j, -len(hist_rb):] = hist_rb.astype(np.int64)
            keep.append(j)
        if not keep:
            continue
        keep = np.asarray(keep, dtype=np.int64)
        batch_users = users[keep].astype(np.int64)
        user_vecs = encode_user_vectors(model, hists[keep], event_types[keep], recency_buckets[keep], batch_size=CFG.retrieval_user_batch_size)
        top_rows, top_scores = ann.query(user_vecs, CFG.ann_oversample)

        for local_row, user_id in enumerate(batch_users):
            # HNSW row id 0 corresponds to item_idx=1.
            row_ids = np.asarray(top_rows[local_row], dtype=np.int64)
            raw_scores = np.asarray(top_scores[local_row], dtype=np.float32)
            item_idxs = row_ids + 1
            valid = (item_idxs > 0) & (item_idxs < len(item_id_by_idx))
            if CFG.filter_seen_context_items:
                seen_idx = set(user_seq.get(int(user_id), []))
                valid &= np.asarray([int(x) not in seen_idx for x in item_idxs], dtype=bool)
            item_idxs = item_idxs[valid]
            raw_scores = raw_scores[valid]
            if len(item_idxs) == 0:
                continue
            # Light user-category/catalog calibration from the user's context history.
            hist_seq = user_seq.get(int(user_id), np.empty(0, dtype=np.int32))
            cat_pref = np.zeros_like(raw_scores, dtype=np.float32)
            catalog_pref = np.zeros_like(raw_scores, dtype=np.float32)
            if len(hist_seq) > 0 and len(cat_l0_arr) > int(np.max(hist_seq, initial=0)):
                hist_cat = cat_l0_arr[hist_seq]
                hist_cat = hist_cat[hist_cat > 0]
                if len(hist_cat) > 0:
                    vals, cnts = np.unique(hist_cat, return_counts=True)
                    pref = {int(v): float(c) / float(len(hist_cat)) for v, c in zip(vals, cnts)}
                    cat_pref = np.asarray([pref.get(int(x), 0.0) for x in cat_l0_arr[item_idxs]], dtype=np.float32)
                hist_catalog = catalog_arr[hist_seq]
                hist_catalog = hist_catalog[hist_catalog > 0]
                if len(hist_catalog) > 0:
                    vals, cnts = np.unique(hist_catalog, return_counts=True)
                    pref = {int(v): float(c) / float(len(hist_catalog)) for v, c in zip(vals, cnts)}
                    catalog_pref = np.asarray([pref.get(int(x), 0.0) for x in catalog_arr[item_idxs]], dtype=np.float32)

            calibrated_scores = (
                float(CFG.bert4rec_raw_score_weight) * raw_scores
                + float(CFG.bert4rec_popularity_prior_weight) * item_pop[item_idxs]
                + float(CFG.bert4rec_user_cat_l0_prior_weight) * cat_pref
                + float(CFG.bert4rec_user_catalog_prior_weight) * catalog_pref
            ).astype(np.float32)
            order = np.argsort(-calibrated_scores, kind="mergesort")
            item_idxs = item_idxs[order][:CFG.candidates_per_user]
            calibrated_scores = calibrated_scores[order][:CFG.candidates_per_user]
            for rank, (item_idx, score) in enumerate(zip(item_idxs, calibrated_scores), start=1):
                item_id = int(item_id_by_idx[int(item_idx)])
                rows_buffer.append((int(user_id), item_id, float(score), int(rank)))

        if len(rows_buffer) >= CFG.retrieval_flush_rows:
            total_rows += write_candidate_buffer(rows_buffer, out_dir, part_id)
            rows_buffer.clear()
            part_id += 1
            gc.collect()

    if rows_buffer:
        total_rows += write_candidate_buffer(rows_buffer, out_dir, part_id)
        rows_buffer.clear()

    meta = {
        "source_name": CFG.source_name,
        "split": split_name,
        "rows": int(total_rows),
        "users": int(len(user_ids)),
        "candidates_per_user": int(CFG.candidates_per_user),
        "ann_oversample": int(CFG.ann_oversample),
        "ann_backend": ann.backend,
        "candidate_inference_time_sec": float(time.perf_counter() - t0),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(CFG),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"{CFG.source_name}/{split_name}: rows={total_rows:,}, time={meta['candidate_inference_time_sec']:.1f}s -> {rel(out_dir)}")
    return {"split": split_name, "status": "built", "rows": int(total_rows), "time_sec": meta["candidate_inference_time_sec"]}

In [8]:
# -----------------------------
# Safe merge with base candidates
# -----------------------------

def normalize_base_lf(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs = []
    if "candidate_rank" not in cols:
        if "rank" in cols:
            exprs.append(pl.col("rank").cast(pl.Int32).alias("candidate_rank"))
        else:
            raise ValueError("Base candidate shard has neither candidate_rank nor rank")
    return lf.with_columns(exprs) if exprs else lf


def normalize_bert4rec_lf(lf: pl.LazyFrame) -> pl.LazyFrame:
    source = CFG.source_name
    cols = set(lazy_columns(lf))
    exprs = []
    if "candidate_rank" not in cols and "rank" in cols:
        exprs.append(pl.col("rank").cast(pl.Int32).alias("candidate_rank"))
    if f"{source}_score" not in cols and "score" in cols:
        exprs.append(pl.col("score").cast(pl.Float32).alias(f"{source}_score"))
    if f"{source}_rank" not in cols and "rank" in cols:
        exprs.append(pl.col("rank").cast(pl.Int32).alias(f"{source}_rank"))
    if f"in_{source}" not in cols:
        exprs.append(pl.lit(1, dtype=pl.Int8).alias(f"in_{source}"))
    if f"f_num__candidate__in_{source}" not in cols:
        exprs.append(pl.lit(1.0, dtype=pl.Float32).alias(f"f_num__candidate__in_{source}"))
    if f"f_num__candidate__{source}_score" not in cols:
        score_expr = pl.col(f"{source}_score") if f"{source}_score" in cols else pl.col("score")
        exprs.append(score_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_score"))
    if f"f_num__candidate__{source}_rank" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append(rank_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_rank"))
    if f"f_num__candidate__{source}_rr" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append((1.0 / rank_expr.cast(pl.Float32).clip(lower_bound=1.0)).cast(pl.Float32).alias(f"f_num__candidate__{source}_rr"))
    return lf.with_columns(exprs) if exprs else lf


def aggregate_merged(df: pl.DataFrame) -> pl.DataFrame:
    agg_exprs = []
    for c in df.columns:
        if c in {"user_id", "item_id"}:
            continue
        if c.startswith("in_") or c.startswith("f_num__candidate__in_") or c in {"_base_present", "_bert4rec_present"}:
            agg_exprs.append(pl.col(c).fill_null(0).max().alias(c))
        elif c.endswith("_rank") or c in {"rank", "candidate_rank", "_base_rank", "_bert4rec_rank", "min_source_rank"}:
            agg_exprs.append(pl.col(c).min().alias(c))
        elif c.endswith("_score") or c.endswith("_rr") or c in {"score", "max_source_score"}:
            agg_exprs.append(pl.col(c).max().alias(c))
        elif c == "source":
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))
        else:
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))

    out = df.group_by(["user_id", "item_id"]).agg(agg_exprs)

    flag_cols = [c for c in out.columns if c.startswith("in_")]
    score_cols = [c for c in out.columns if c.endswith("_score") and not c.startswith("f_num__")]
    rank_cols = [c for c in out.columns if c.endswith("_rank") and not c.startswith("f_num__")]
    exprs = []
    if flag_cols:
        exprs.append(pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in flag_cols]).cast(pl.Int16).alias("sources_count"))
    if score_cols:
        exprs.append(pl.max_horizontal(*[pl.col(c).fill_null(-1e9).cast(pl.Float32) for c in score_cols]).cast(pl.Float32).alias("max_source_score"))
    if rank_cols:
        exprs.append(pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000).cast(pl.Int32) for c in rank_cols]).cast(pl.Int32).alias("min_source_rank"))
    if exprs:
        out = out.with_columns(exprs)

    # Base candidates are kept first; BERT4Rec is appended. This prevents a weak DL source from replacing strong BM25/ALS candidates.
    out = out.with_columns(
        pl.when(pl.col("_base_present").fill_null(0) > 0).then(0).otherwise(1).alias("_priority"),
        pl.when(pl.col("_base_present").fill_null(0) > 0)
        .then(pl.col("_base_rank").fill_null(1_000_000_000))
        .otherwise(pl.col("_bert4rec_rank").fill_null(1_000_000_000))
        .cast(pl.Int32)
        .alias("_sort_rank"),
    )
    out = (
        out.sort(["user_id", "_priority", "_sort_rank", "item_id"])
        .with_columns((pl.int_range(0, pl.len()).over("user_id") + 1).cast(pl.Int32).alias("candidate_rank"))
        .filter(pl.col("candidate_rank") <= CFG.merged_total_keep_per_user)
        .drop(["_priority", "_sort_rank"])
    )
    return out


def safe_merge_split(split: str) -> dict:
    base_dir = resolve_base_candidate_dir(split)
    bert4rec_dir = CANDIDATES_DIR / CFG.source_name / split
    out_dir = CANDIDATES_DIR / CFG.output_merged_name / split

    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_merged:
        print(f"{split}: merged reused -> {rel(out_dir)}")
        return {"split": split, "status": "reused", "out_dir": rel(out_dir)}

    if not candidate_dir_looks_complete(bert4rec_dir):
        raise FileNotFoundError(f"BERT4Rec source not found for split={split}: {bert4rec_dir}")

    clean_output_dir(out_dir)
    rows_total = 0
    for shard_id in trange(CFG.n_shards, desc=f"Safe merge {split}"):
        base_lf = scan_candidate_shard_or_none(base_dir, shard_id)
        sas_lf = scan_candidate_shard_or_none(bert4rec_dir, shard_id)
        if base_lf is None:
            continue

        base_lf = normalize_base_lf(base_lf).filter(pl.col("candidate_rank") <= CFG.base_keep_per_user).with_columns(
            pl.lit(1, dtype=pl.Int8).alias("_base_present"),
            pl.lit(0, dtype=pl.Int8).alias("_bert4rec_present"),
            pl.col("candidate_rank").cast(pl.Int32).alias("_base_rank"),
            pl.lit(None, dtype=pl.Int32).alias("_bert4rec_rank"),
        )

        frames = [base_lf]
        if sas_lf is not None:
            # Filter BERT4Rec users to users from the base shard.
            base_users = base_lf.select(pl.col("user_id").cast(pl.Int64).unique())
            sas_lf = (
                normalize_bert4rec_lf(sas_lf)
                .join(base_users, on="user_id", how="inner")
                .filter(pl.col("rank").cast(pl.Int32) <= CFG.bert4rec_keep_per_user)
                .with_columns(
                    pl.lit(0, dtype=pl.Int8).alias("_base_present"),
                    pl.lit(1, dtype=pl.Int8).alias("_bert4rec_present"),
                    pl.lit(None, dtype=pl.Int32).alias("_base_rank"),
                    pl.col("rank").cast(pl.Int32).alias("_bert4rec_rank"),
                )
            )
            frames.append(sas_lf)

        merged = collect_streaming(pl.concat(frames, how="diagonal_relaxed"))
        if merged.is_empty():
            continue
        out = aggregate_merged(merged)
        sdir = out_dir / f"user_shard={shard_id}"
        ensure_dir(sdir)
        out.write_parquet(sdir / "part-00000.parquet", compression="zstd")
        rows_total += out.height
        del merged, out
        gc.collect()

    meta = {
        "split": split,
        "source_name": CFG.source_name,
        "base_dir": rel(base_dir),
        "bert4rec_dir": rel(bert4rec_dir),
        "out_dir": rel(out_dir),
        "rows": int(rows_total),
        "base_keep_per_user": int(CFG.base_keep_per_user),
        "bert4rec_keep_per_user": int(CFG.bert4rec_keep_per_user),
        "merged_total_keep_per_user": int(CFG.merged_total_keep_per_user),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(CFG),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"{split}: merged rows={rows_total:,} -> {rel(out_dir)}")
    return {"split": split, "status": "built", "rows": int(rows_total), "out_dir": rel(out_dir)}

In [9]:
# -----------------------------
# Validation metrics
# -----------------------------

def _unique_int_list(values) -> list[int]:
    if values is None:
        return []
    return list(dict.fromkeys(int(x) for x in values))


def hit_rate_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(any(int(x) in gt for x in pred_items[:k])) if gt else np.nan


def recall_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(len(gt & set(int(x) for x in pred_items[:k])) / len(gt)) if gt else np.nan


def ndcg_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return float(dcg / idcg) if idcg else 0.0


def ap_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    hits = 0
    score = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            hits += 1
            score += hits / rank
    return float(score / len(gt))


def rr_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            return float(1.0 / rank)
    return 0.0


def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .filter(pl.col("cnt_orders").fill_null(0) > 0)
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_id"])
    )
    result: dict[int, list[int]] = defaultdict(list)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].append(int(i))
    return dict(result)


def evaluate_candidate_dir(candidates_dir: Path, ground_truth: dict[int, list[int]], *, ks: Iterable[int], rank_col: str, label: str) -> pd.DataFrame:
    if not path_has_parquet(candidates_dir):
        return pd.DataFrame()
    ks = tuple(int(k) for k in ks)
    sums = {k: defaultdict(float) for k in ks}
    candidate_counts = []
    users_eval = 0
    unique_items: set[int] = set()

    for shard_id in trange(CFG.n_shards, desc=f"Evaluate {label}"):
        lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("rank" if "rank" in cols else "candidate_rank")
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(actual_rank_col).cast(pl.Int32).alias("eval_rank"),
            ).sort(["user_id", "eval_rank", "item_id"])
        )
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for user_id, group in pdf.groupby("user_id", sort=False):
            user_id = int(user_id)
            gt = ground_truth.get(user_id)
            if not gt:
                continue
            pred = group["item_id"].astype(np.int64).tolist()
            users_eval += 1
            candidate_counts.append(len(pred))
            unique_items.update(pred[:max(ks)])
            for k in ks:
                sums[k]["hit_rate"] += hit_rate_at_k(gt, pred, k)
                sums[k]["recall"] += recall_at_k(gt, pred, k)
                sums[k]["ndcg"] += ndcg_at_k(gt, pred, k)
                sums[k]["map"] += ap_at_k(gt, pred, k)
                sums[k]["mrr"] += rr_at_k(gt, pred, k)
        del pdf, part
        gc.collect()

    rows = []
    for k in ks:
        denom = max(1, users_eval)
        rows.append({
            "label": label,
            "k": k,
            "users_eval": int(users_eval),
            "hit_rate": sums[k]["hit_rate"] / denom,
            "recall": sums[k]["recall"] / denom,
            "ndcg": sums[k]["ndcg"] / denom,
            "map": sums[k]["map"] / denom,
            "mrr": sums[k]["mrr"] / denom,
            "avg_candidates_per_user": float(np.mean(candidate_counts)) if candidate_counts else 0.0,
            "p50_candidates_per_user": float(np.median(candidate_counts)) if candidate_counts else 0.0,
            "unique_items_at_k": int(len(unique_items)),
        })
    return pd.DataFrame(rows)


def build_positive_pairs_df(gt: dict[int, list[int]]) -> pl.DataFrame:
    rows = [(int(u), int(i)) for u, items in gt.items() for i in items]
    return pl.DataFrame(rows, schema=["user_id", "item_id"], orient="row").with_columns(
        pl.col("user_id").cast(pl.Int64),
        pl.col("item_id").cast(pl.Int64),
    )


def collect_found_positive_pairs_at_k(candidates_dir: Path, gt_pairs: pl.DataFrame, k: int, rank_col: str) -> pl.DataFrame:
    parts = []
    gt_lf = gt_pairs.lazy()
    for shard_id in trange(CFG.n_shards, desc=f"Positive pairs {candidates_dir.name}@{k}"):
        lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("rank" if "rank" in cols else "candidate_rank")
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(actual_rank_col).cast(pl.Int32).alias("rank"),
            )
            .filter(pl.col("rank") <= int(k))
            .join(gt_lf, on=["user_id", "item_id"], how="inner")
            .select("user_id", "item_id")
            .unique()
        )
        if not part.is_empty():
            parts.append(part)
    return pl.concat(parts, how="vertical_relaxed").unique() if parts else pl.DataFrame({"user_id": [], "item_id": []}, schema={"user_id": pl.Int64, "item_id": pl.Int64})


def quota_ablation(gt_pairs: pl.DataFrame, base_dir: Path, bert4rec_dir: Path, quotas: tuple[int, ...] = (50, 100, 200, 300)) -> pd.DataFrame:
    base_pos = collect_found_positive_pairs_at_k(base_dir, gt_pairs, CFG.base_keep_per_user, "candidate_rank")
    rows = []
    total = max(1, gt_pairs.height)
    for q in quotas:
        sas_pos = collect_found_positive_pairs_at_k(bert4rec_dir, gt_pairs, q, "rank")
        unique = sas_pos.join(base_pos, on=["user_id", "item_id"], how="anti")
        union = pl.concat([base_pos, sas_pos], how="vertical_relaxed").unique()
        rows.append({
            "base_keep": int(CFG.base_keep_per_user),
            "bert4rec_quota": int(q),
            "total_cap": int(CFG.base_keep_per_user + q),
            "base_found_pairs": int(base_pos.height),
            "bert4rec_found_pairs": int(sas_pos.height),
            "bert4rec_unique_vs_base_pairs": int(unique.height),
            "union_found_pairs": int(union.height),
            "incremental_pair_recall": float(unique.height / total),
            "union_pair_recall": float(union.height / total),
        })
    return pd.DataFrame(rows)

## Запуск

Этот ноутбук не трогает non-DL candidates: он читает готовый `data/datamart/candidates/merged`.

Для первого запуска оставь defaults. Если уже есть частичный неудачный `dl_bert4rec_small`, включи clean rebuild в config-ячейке.

После успешного запуска для реранкера используй `candidate_input_name = "merged_with_bert4rec_small"` и `rebuild_datasets=True`.

In [10]:
# -----------------------------
# Run pipeline
# -----------------------------

# 1. Train or load checkpoint.
model, data = train_or_load_bert4rec(CFG.checkpoint_split)
model = model.to(DEVICE)

# 2. Build or load ANN index for checkpoint item vectors.
ann = build_or_load_ann(model, data["item_id_by_idx"])

# 3. Generate BERT4Rec sources for all configured splits.
build_rows = []
for split in CFG.splits_to_build_sources:
    build_rows.append(generate_bert4rec_candidates_for_split(model, ann, split, data["item2idx"], data["item_id_by_idx"], data["item_weight_by_idx"]))
build_report = pd.DataFrame(build_rows)
display(build_report)
build_report.to_parquet(REPORTS_DIR / "dl_bert4rec_small_build_report.parquet", index=False)
build_report.to_csv(REPORTS_DIR / "dl_bert4rec_small_build_report.csv", index=False)

# 4. Safe merge with non-DL base candidates.
merge_rows = []
for split in CFG.splits_to_merge:
    merge_rows.append(safe_merge_split(split))
merge_report = pd.DataFrame(merge_rows)
display(merge_report)
merge_report.to_parquet(REPORTS_DIR / "dl_bert4rec_small_merge_report.parquet", index=False)
merge_report.to_csv(REPORTS_DIR / "dl_bert4rec_small_merge_report.csv", index=False)

# 5. Validation metrics and unique positives.
if CFG.evaluate_val:
    gt_val = build_ground_truth(CFG.val_target_start, CFG.val_target_end)
    gt_pairs = build_positive_pairs_df(gt_val)

    base_val = CANDIDATES_DIR / CFG.base_candidate_name / "val"
    bert4rec_val = CANDIDATES_DIR / CFG.source_name / "val"
    merged_val = CANDIDATES_DIR / CFG.output_merged_name / "val"

    frames = [
        evaluate_candidate_dir(base_val, gt_val, ks=CFG.metric_ks, rank_col="candidate_rank", label="base_non_dl"),
        evaluate_candidate_dir(bert4rec_val, gt_val, ks=CFG.metric_ks, rank_col="rank", label=CFG.source_name),
        evaluate_candidate_dir(merged_val, gt_val, ks=CFG.metric_ks, rank_col="candidate_rank", label=CFG.output_merged_name),
    ]
    val_metrics = pd.concat([f for f in frames if f is not None and not f.empty], ignore_index=True)
    display(val_metrics)
    val_metrics.to_parquet(REPORTS_DIR / "dl_bert4rec_small_val_metrics.parquet", index=False)
    val_metrics.to_csv(REPORTS_DIR / "dl_bert4rec_small_val_metrics.csv", index=False)

    base_pos = collect_found_positive_pairs_at_k(base_val, gt_pairs, CFG.base_keep_per_user, "candidate_rank")
    sas_pos = collect_found_positive_pairs_at_k(bert4rec_val, gt_pairs, CFG.bert4rec_keep_per_user, "rank")
    merged_pos = collect_found_positive_pairs_at_k(merged_val, gt_pairs, CFG.merged_total_keep_per_user, "candidate_rank")
    unique = sas_pos.join(base_pos, on=["user_id", "item_id"], how="anti")
    overlap = sas_pos.join(base_pos, on=["user_id", "item_id"], how="inner")

    unique_report = pd.DataFrame([{
        "target_positive_pairs": int(gt_pairs.height),
        "base_found_pairs": int(base_pos.height),
        "bert4rec_found_pairs": int(sas_pos.height),
        "bert4rec_unique_vs_base_pairs": int(unique.height),
        "bert4rec_overlap_with_base_pairs": int(overlap.height),
        "merged_found_pairs": int(merged_pos.height),
        "base_pair_recall": float(base_pos.height / max(1, gt_pairs.height)),
        "bert4rec_pair_recall": float(sas_pos.height / max(1, gt_pairs.height)),
        "bert4rec_unique_pair_recall": float(unique.height / max(1, gt_pairs.height)),
        "merged_pair_recall": float(merged_pos.height / max(1, gt_pairs.height)),
    }])
    display(unique_report)
    unique_report.to_parquet(REPORTS_DIR / "dl_bert4rec_small_unique_positives.parquet", index=False)
    unique_report.to_csv(REPORTS_DIR / "dl_bert4rec_small_unique_positives.csv", index=False)

    quota = quota_ablation(gt_pairs, base_val, bert4rec_val)
    display(quota)
    quota.to_parquet(REPORTS_DIR / "dl_bert4rec_small_quota_ablation.parquet", index=False)
    quota.to_csv(REPORTS_DIR / "dl_bert4rec_small_quota_ablation.csv", index=False)

# 6. Manifest.
manifest = {
    "source_name": CFG.source_name,
    "output_merged_name": CFG.output_merged_name,
    "checkpoint_path": rel(checkpoint_path(CFG.checkpoint_split)),
    "candidate_dirs": {
        split: rel(CANDIDATES_DIR / CFG.source_name / split)
        for split in CFG.splits_to_build_sources
    },
    "merged_dirs": {
        split: rel(CANDIDATES_DIR / CFG.output_merged_name / split)
        for split in CFG.splits_to_merge
    },
    "reports": {
        "build_report": rel(REPORTS_DIR / "dl_bert4rec_small_build_report.parquet"),
        "merge_report": rel(REPORTS_DIR / "dl_bert4rec_small_merge_report.parquet"),
        "val_metrics": rel(REPORTS_DIR / "dl_bert4rec_small_val_metrics.parquet"),
        "unique_positives": rel(REPORTS_DIR / "dl_bert4rec_small_unique_positives.parquet"),
        "quota_ablation": rel(REPORTS_DIR / "dl_bert4rec_small_quota_ablation.parquet"),
    },
    "config": asdict(CFG),
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
}
manifest_path = CANDIDATES_DIR / "manifest_dl_bert4rec_small.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("manifest saved:", rel(manifest_path))
print("Next step: run reranker on candidate_input_name =", CFG.output_merged_name)

val: building context interactions 2025-04-18..2025-06-17
val: interactions rows=94,605,512


Build user sequences:   0%|          | 0/910102 [00:00<?, ?it/s]

val: training sequences saved -> data\processed\dl_candidate_generators_bert4rec_small\sequences\val


Read item catalog mapping:   0%|          | 0/100 [00:00<?, ?it/s]

val: item side arrays saved -> data\processed\dl_candidate_generators_bert4rec_small\item_side\item_side_val.npz
Training BERT4Rec checkpoint for split=val


C:\Users\denis\AppData\Local\Temp\ipykernel_31040\999585078.py:280: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


... (53 outputs left out)

Positive pairs val@200:   0%|          | 0/128 [00:00<?, ?it/s]

Positive pairs val@300:   0%|          | 0/128 [00:00<?, ?it/s]

,base_keep,bert4rec_quota,total_cap,base_found_pairs,bert4rec_found_pairs,bert4rec_unique_vs_base_pairs,union_found_pairs,incremental_pair_recall,union_pair_recall
0,700,50,750,140798,4951,1952,142750,0.001238,0.090536
1,700,100,800,140798,7088,3331,144129,0.002113,0.091411
2,700,200,900,140798,9961,5401,146199,0.003425,0.092724
3,700,300,1000,140798,11933,6865,147663,0.004354,0.093652


manifest saved: data\datamart\candidates\manifest_dl_bert4rec_small.json
Next step: run reranker on candidate_input_name = merged_with_bert4rec_small
